In [ ]:
import os; os.environ["CUDA_LAUNCH_BLOCKING"]="1"
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
import torch
import numpy as np

from nocode_robot_programming.state_decision.plots import pretty_confusion_matrix
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda

seed = 50
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

from nocode_robot_programming.state_decision_dataset_prepare.dataset_auto import TrajectoryDatasetEvaluationViewBuilder
dataset_builder = TrajectoryDatasetEvaluationViewBuilder()
datasets = dataset_builder.load_eval_from_task("petr_kin_peg_pick")

# 1. Evaluation of AE-GP method on Dataset `d1`

- We have `datafileloader` that loads recording files from memory
- We have `datasets` for peg pick `task` extracted with given settings
- Now we can see the dataset images:

In [ ]:
d_train, d_test, d_text = datasets[0]
# d_train.showcase(fps=20)
# d_train.showcase_captions(fps=20)
d_train.play_video(fps=2, scale=1.0)
# d_test.play_video(fps=10)

In [ ]:
# for task_name in dataset.tasks.keys():
for d_train, d_test, d_text in datasets:
    print(d_text)
    decider = AEGP(binary=True, pix=64)
    decider.train(d_train.X, d_train.y_int, d_train.y_cls)

    y_pred = decider.predict_many(d_train.X); ipt.save()
    pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False, figsize=None); ipt.save()

    y_pred = decider.predict_many(d_test.X); ipt.save()
    pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False, figsize=None); ipt.save()

    ipt.show()

    rr = range(0,d_train.X.size(0),5) # reconstructed range - image selections
    reconstructed_images = decider.videoembedder.model.forward(decider.tf(d_train.X[rr]).unsqueeze(1)).squeeze()
    display(show_gray_video_cuda(torch.concatenate([decider.tf(d_train.X[rr]), reconstructed_images], dim=2), fps=20, scale=5))

# Conclusion:

- I managed to train the AE. GP 100% on train, but ~50% on test data. What am I missing?